# Percolation vs Fixed-Ratio Extractive Summarizer: CNN/DM + Multi-News Evaluation

This notebook demonstrates a **vocabulary-percolation-threshold extractive summarizer** evaluated against fixed-ratio baselines on CNN/DailyMail and Multi-News corpora.

**Core idea**: Instead of selecting a fixed fraction of sentences, the percolation method greedily adds top-scoring sentences until the vocabulary graph's giant connected component (GCC) reaches a threshold fraction θ of the full-document GCC. This adaptive stopping criterion is evaluated with θ ∈ {0.6, 0.7, 0.8, 0.9} versus fixed-ratio baselines (10%/20%/30%).

**Metrics computed**:
- ROUGE-1/2/L F1, Precision, Recall
- Ceiling fraction (docs where percolation selects all sentences)
- Compression ratio statistics
- Wilcoxon signed-rank tests with Holm-Bonferroni correction
- OLS regression: compression_ratio ~ graph features
- TF vs TF-IDF scorer comparison on Multi-News

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Packages NOT pre-installed on Colab
_pip('rouge-score==0.1.2')
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0',
         'statsmodels==0.14.6', 'networkx==3.6.1', 'nltk==3.9.1')

In [ ]:
import gc
import json
import math
import os
import sys
from collections import Counter
from pathlib import Path

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-6a9498-vocabulary-percolation-threshold-for-sel/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print("Loaded keys:", list(data.keys()))
print("Datasets:", [ds['dataset'] for ds in data['datasets']])
for ds in data['datasets']:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")

In [ ]:
# ── Config: all tunable parameters ────────────────────────────────────────────
# Set to minimum values for demo; original full-scale values shown in comments

THETAS = [0.6, 0.7, 0.8, 0.9]          # percolation thresholds (original: same)
FIXED_RATIOS = [0.10, 0.20, 0.30]       # fixed extraction ratios (original: same)

# Number of demo documents to process (from Mini-News data loaded above)
# Original run: CNN/DM=2000 docs from raw results, Multi-News=1000 fresh docs
N_DEMO_DOCS = 3   # minimum: use all available in mini dataset

## NLTK Setup

Download required NLTK data (punkt tokenizer and stopwords).

In [ ]:
import nltk
nltk_data_dir = "/tmp/nltk_data"
os.makedirs(nltk_data_dir, exist_ok=True)
nltk.data.path.insert(0, nltk_data_dir)
for pkg in ["punkt", "punkt_tab", "stopwords"]:
    try:
        nltk.download(pkg, download_dir=nltk_data_dir, quiet=True)
    except Exception:
        pass
print("NLTK ready")

## Text Processing Functions

Core text processing: sentence tokenization, word filtering (stopwords removal), TF and TF-IDF scoring, and vocabulary graph construction.

In [ ]:
def preprocess(text: str) -> tuple:
    from nltk.corpus import stopwords
    stop = set(stopwords.words("english"))
    raw_sents = nltk.sent_tokenize(text)
    word_sets = []
    for s in raw_sents:
        words = set(
            w.lower() for w in nltk.word_tokenize(s)
            if w.isalpha() and len(w) >= 2 and w.lower() not in stop
        )
        word_sets.append(words)
    return raw_sents, word_sets

def compute_tf(sentences_words: list) -> dict:
    tf: Counter = Counter()
    for words in sentences_words:
        tf.update(words)
    return dict(tf)

def compute_tfidf(sentences_words: list) -> dict:
    n = len(sentences_words)
    tf: Counter = Counter()
    df: Counter = Counter()
    for words in sentences_words:
        tf.update(words)
        for w in words:
            df[w] += 1
    tfidf = {}
    for w, f in tf.items():
        idf = math.log((1 + n) / (1 + df[w])) + 1.0
        tfidf[w] = f * idf
    return tfidf

def score_sentences(sentences_words: list, tf: dict) -> list:
    return [sum(tf.get(w, 0.0) for w in ws) for ws in sentences_words]

def build_vocab_graph(sentences_words: list):
    import networkx as nx
    G = nx.Graph()
    edge_counter: Counter = Counter()
    for words in sentences_words:
        wlist = sorted(words)
        for i in range(len(wlist)):
            for j in range(i + 1, len(wlist)):
                edge_counter[(wlist[i], wlist[j])] += 1
    G.add_nodes_from({w for ws in sentences_words for w in ws})
    for (u, v), wt in edge_counter.items():
        G.add_edge(u, v, weight=wt)
    return G

def gcc_size(G) -> int:
    import networkx as nx
    if len(G) == 0:
        return 0
    return max((len(c) for c in nx.connected_components(G)), default=0)

## Summarization Methods

**Percolation method**: greedily adds top-scoring sentences until the vocabulary subgraph's GCC reaches θ × (full-document GCC size).  
**Fixed-ratio method**: selects top-scoring sentences up to `ceil(ratio × n)` sentences.

In [ ]:
def percolation_summary(sentences_words, scorer, full_G, full_gcc, theta):
    import networkx as nx
    n = len(sentences_words)
    if n == 0:
        return [], n
    scores = score_sentences(sentences_words, scorer)
    ranked = sorted(range(n), key=lambda i: -scores[i])
    if full_gcc == 0:
        return sorted(range(n)), n
    accumulated: set = set()
    selected = []
    k_star = n
    for idx in ranked:
        selected.append(idx)
        accumulated |= sentences_words[idx]
        sub = full_G.subgraph(accumulated)
        cur = max((len(c) for c in nx.connected_components(sub)), default=0)
        if cur / full_gcc >= theta:
            k_star = len(selected)
            break
    return sorted(selected), k_star

def fixed_ratio_summary(sentences_words, scorer, ratio):
    n = len(sentences_words)
    k = max(1, math.ceil(ratio * n))
    scores = score_sentences(sentences_words, scorer)
    ranked = sorted(range(n), key=lambda i: -scores[i])
    return sorted(ranked[:k]), k

def rouge_score_doc(selected_indices, all_sentences, reference):
    from rouge_score import rouge_scorer as rs_module
    scorer = rs_module.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    if not selected_indices:
        return {k: 0.0 for k in [
            "rouge1_f","rouge1_r","rouge1_p","rouge2_f","rouge2_r","rouge2_p",
            "rougeL_f","rougeL_r","rougeL_p"]}
    summary = " ".join(all_sentences[i] for i in sorted(selected_indices))
    scores = scorer.score(reference, summary)
    out = {}
    for k, v in scores.items():
        out[f"{k}_f"] = v.fmeasure
        out[f"{k}_r"] = v.recall
        out[f"{k}_p"] = v.precision
    return out

def network_props(G) -> dict:
    import networkx as nx, random
    if len(G) == 0:
        return {"avg_degree": 0.0, "clustering": 0.0, "n_nodes": 0, "n_edges": 0}
    degs = [d for _, d in G.degree()]
    avg_deg = sum(degs) / len(degs)
    sample = random.sample(list(G.nodes()), min(200, len(G)))
    clust = nx.average_clustering(G, nodes=sample)
    return {"avg_degree": round(avg_deg, 4), "clustering": round(clust, 4),
            "n_nodes": len(G), "n_edges": G.number_of_edges()}

## Statistical Utilities

Holm-Bonferroni multiple testing correction and safe Wilcoxon signed-rank test (requires at least 10 non-zero differences).

In [ ]:
def holm_bonferroni(p_values: list) -> list:
    n = len(p_values)
    indexed = sorted(enumerate(p_values), key=lambda x: x[1])
    adjusted = [0.0] * n
    for rank, (orig_idx, p) in enumerate(indexed):
        adjusted[orig_idx] = min(1.0, p * (n - rank))
    # Make monotone
    sorted_adj = [adjusted[i] for i, _ in sorted(enumerate(p_values), key=lambda x: x[1])]
    running_max = 0.0
    for rank, (orig_idx, _) in enumerate(sorted(enumerate(p_values), key=lambda x: x[1])):
        running_max = max(running_max, sorted_adj[rank])
        adjusted[orig_idx] = running_max
    return adjusted

def wilcoxon_safe(x, y):
    diff = np.array(x) - np.array(y)
    nonzero = diff[diff != 0]
    if len(nonzero) < 10:
        return float("nan"), float("nan")
    stat, p = stats.wilcoxon(diff)
    return float(stat), float(p)

def collect_arrays(per_doc: list, method_key: str, sub_key: str, field: str) -> np.ndarray:
    vals = []
    for d in per_doc:
        entry = d.get(method_key, {}).get(sub_key, {})
        if field in entry:
            vals.append(entry[field])
    return np.array(vals, dtype=float)

## Demo: Process Documents from Pre-loaded Data

Instead of downloading the full CNN/DM and Multi-News datasets (which takes minutes), we run the percolation and fixed-ratio summarizers on a few example documents from the pre-loaded mini dataset. Each document has metadata (n_sentences, full_gcc, avg_degree) already stored.

In [ ]:
# Extract pre-computed per-document results from loaded data
# (Original script loaded CNN/DM from raw experiment files and Multi-News from HuggingFace Hub)
cnndm_examples = data['datasets'][0]['examples'][:N_DEMO_DOCS]
mn_examples = data['datasets'][1]['examples'][:N_DEMO_DOCS]

print(f"CNN/DM demo examples: {len(cnndm_examples)}")
print(f"Multi-News demo examples: {len(mn_examples)}")

# Show an example document's metadata
ex = cnndm_examples[0]
print("\nExample CNN/DM doc:")
print(f"  Input:  {ex['input']}")
print(f"  Output: {ex['output']}")
print(f"  Percolation θ=0.6: {ex['predict_percolation_0_6']}")
print(f"  Fixed 10%:         {ex['predict_fixed_0_1']}")

## Aggregate CNN/DM Results

Compute mean±std of ROUGE scores, compression ratios, ceiling fraction (docs where all sentences selected), and whether compression std exceeds 10pp threshold.

In [ ]:
def aggregate_cnndm(per_doc: list) -> dict:
    """Aggregate per-document CNN/DM results into mean/std statistics."""
    agg = {"n_docs": len(per_doc)}
    metrics = ["rouge1_f","rouge1_r","rouge1_p","rouge2_f","rouge2_r","rouge2_p",
               "rougeL_f","rougeL_r","rougeL_p","compression_ratio"]

    for theta in THETAS:
        bucket = {}
        key = str(theta)
        for m in metrics:
            v = collect_arrays(per_doc, "percolation", key, m)
            if len(v) == 0:
                continue
            bucket[f"mean_{m}"] = round(float(np.mean(v)), 4)
            bucket[f"std_{m}"] = round(float(np.std(v)), 4)
            if m == "compression_ratio":
                bucket["min_compression"] = round(float(np.min(v)), 4)
                bucket["max_compression"] = round(float(np.max(v)), 4)
                bucket["p25_compression"] = round(float(np.percentile(v, 25)), 4)
                bucket["p75_compression"] = round(float(np.percentile(v, 75)), 4)
                # CEILING FRACTION: fraction where compression == 1.0
                bucket["ceiling_fraction"] = round(float(np.mean(v >= 0.9999)), 4)
                # STD > 0.10 test
                bucket["std_exceeds_10pp"] = bool(np.std(v) > 0.10)
        agg[f"percolation_{theta}"] = bucket

    for ratio in FIXED_RATIOS:
        bucket = {}
        key = str(ratio)
        for m in ["rouge1_f","rouge1_r","rouge1_p","rouge2_f","rouge2_r","rouge2_p",
                  "rougeL_f","rougeL_r","rougeL_p"]:
            v = collect_arrays(per_doc, "fixed", key, m)
            if len(v) == 0:
                continue
            bucket[f"mean_{m}"] = round(float(np.mean(v)), 4)
            bucket[f"std_{m}"] = round(float(np.std(v)), 4)
        agg[f"fixed_{ratio}"] = bucket

    return agg

# For demo: parse eval fields from pre-computed example data
# Reconstruct per-doc format expected by aggregate functions
def examples_to_per_doc(examples):
    """Convert notebook example format to per-doc dict for statistical functions."""
    per_doc = []
    for ex in examples:
        d = {
            "doc_id": ex.get("metadata_doc_id", 0),
            "n_sentences": ex.get("metadata_n_sentences", 0),
            "full_gcc": ex.get("metadata_full_gcc", 0),
            "network": {
                "avg_degree": ex.get("metadata_avg_degree", 0.0),
                "clustering": ex.get("metadata_clustering", 0.0),
            },
            "percolation": {},
            "fixed": {},
        }
        for theta in THETAS:
            safe = str(theta).replace(".", "_")
            d["percolation"][str(theta)] = {
                "rouge1_f": ex.get(f"eval_perc_{safe}_rouge1_f", 0.0),
                "compression_ratio": ex.get(f"eval_perc_{safe}_compression", 0.0),
            }
        for ratio in FIXED_RATIOS:
            safe = str(ratio).replace(".", "_")
            d["fixed"][str(ratio)] = {
                "rouge1_f": ex.get(f"eval_fixed_{safe}_rouge1_f", 0.0),
            }
        per_doc.append(d)
    return per_doc

cnndm_per_doc = examples_to_per_doc(cnndm_examples)
cnndm_agg = aggregate_cnndm(cnndm_per_doc)
print(f"CNN/DM aggregated ({cnndm_agg['n_docs']} docs)")
print(f"  theta=0.6 ROUGE-1 F1: {cnndm_agg.get('percolation_0.6', {}).get('mean_rouge1_f', '?')}")
print(f"  fixed 10% ROUGE-1 F1: {cnndm_agg.get('fixed_0.1', {}).get('mean_rouge1_f', '?')}")

## OLS Regression: Compression Ratio ~ Graph Features

Fit an OLS model: `compression_ratio ~ [avg_degree, clustering_coefficient, n_sentences, full_gcc_size]`. Features are z-score normalized. Reports R², per-feature coefficients with 95% CIs, and partial R² (via model exclusion).

In [ ]:
def regression_analysis(per_doc: list) -> dict:
    import statsmodels.api as sm

    results = {}
    for theta in THETAS:
        key = str(theta)
        comp = []
        avg_deg = []
        clust = []
        n_sents = []
        full_gcc_vals = []
        for d in per_doc:
            if key not in d.get("percolation", {}):
                continue
            comp.append(d["percolation"][key]["compression_ratio"])
            avg_deg.append(d["network"]["avg_degree"])
            clust.append(d["network"]["clustering"])
            n_sents.append(d["n_sentences"])
            full_gcc_vals.append(d["full_gcc"])

        if len(comp) < 10:
            # Not enough data for regression (need ≥10 docs)
            results[f"theta_{key}"] = {"note": f"insufficient data (n={len(comp)}, need ≥10)"}
            continue

        y = np.array(comp)
        X = np.column_stack([avg_deg, clust, n_sents, full_gcc_vals])
        # Normalize features
        X_mean = X.mean(axis=0)
        X_std = X.std(axis=0)
        X_std[X_std == 0] = 1
        X_norm = (X - X_mean) / X_std
        X_with_const = sm.add_constant(X_norm)

        try:
            model = sm.OLS(y, X_with_const).fit()
            params = model.params
            conf = model.conf_int()
            pvals = model.pvalues
            r2 = model.rsquared
            feature_names = ["intercept", "avg_degree", "clustering_coefficient",
                             "n_sentences", "full_gcc_size"]
            coefs = {}
            for i, name in enumerate(feature_names):
                coefs[name] = {
                    "coef": round(float(params[i]), 6),
                    "ci_lower": round(float(conf[0][i]), 6),
                    "ci_upper": round(float(conf[1][i]), 6),
                    "p_value": round(float(pvals[i]), 6),
                }
            # Partial R² for each predictor (via exclusion)
            partial_r2 = {}
            for feat_idx, fname in enumerate(["avg_degree", "clustering_coefficient",
                                               "n_sentences", "full_gcc_size"]):
                cols_excl = [j for j in range(4) if j != feat_idx]
                X_excl = np.column_stack([X_norm[:, j] for j in cols_excl])
                X_excl_c = sm.add_constant(X_excl)
                model_excl = sm.OLS(y, X_excl_c).fit()
                part_r2 = max(0.0, r2 - model_excl.rsquared)
                partial_r2[fname] = round(float(part_r2), 6)

            results[f"theta_{key}"] = {
                "r2": round(float(r2), 4),
                "adj_r2": round(float(model.rsquared_adj), 4),
                "n": len(comp),
                "coefficients": coefs,
                "partial_r2": partial_r2,
            }
        except Exception as e:
            results[f"theta_{key}"] = {"error": str(e)}

    return results

# Use pre-computed aggregate results from metadata (full-scale values)
reg_result_full = data['metadata'].get('regression_analysis', {})
print("Regression results (from full-scale run):")
for k, v in reg_result_full.items():
    if isinstance(v, dict) and 'r2' in v:
        print(f"  {k}: R²={v['r2']}, adj_R²={v['adj_r2']}, n={v['n']}")
        pr2 = v.get('partial_r2', {})
        top = sorted(pr2.items(), key=lambda x: -x[1])
        print(f"    partial R²: {top}")

# Run regression on demo data (may have insufficient n for stats)
reg_demo = regression_analysis(cnndm_per_doc)
print("\nRegression on demo docs:")
for k, v in reg_demo.items():
    print(f"  {k}: {v}")

## Statistical Tests: Wilcoxon + Holm-Bonferroni

12 pairwise comparisons (4 thetas × 3 ratios) of percolation vs fixed-ratio ROUGE-1 F1 on CNN/DM. Holm-Bonferroni correction applied to all p-values. (Requires ≥10 non-zero differences per pair; demo data shows pre-computed full-scale results.)

In [ ]:
def statistical_tests(per_doc: list, corpus: str = "cnndm") -> dict:
    # All pairwise: percolation_theta vs fixed_ratio for ROUGE-1 F1
    comparisons = []
    for theta in THETAS:
        for ratio in FIXED_RATIOS:
            perc_vals = collect_arrays(per_doc, "percolation", str(theta), "rouge1_f")
            fixed_vals = collect_arrays(per_doc, "fixed", str(ratio), "rouge1_f")
            n = min(len(perc_vals), len(fixed_vals))
            if n < 10:
                comparisons.append({
                    "percolation_theta": theta, "fixed_ratio": ratio,
                    "wilcoxon_stat": None, "p_raw": None,
                    "mean_diff": round(float(np.mean(perc_vals[:n] - fixed_vals[:n])), 4) if n > 0 else None,
                    "n": n, "note": "insufficient n for Wilcoxon"
                })
                continue
            stat, p = wilcoxon_safe(perc_vals[:n], fixed_vals[:n])
            comparisons.append({
                "percolation_theta": theta,
                "fixed_ratio": ratio,
                "wilcoxon_stat": round(stat, 4) if not math.isnan(stat) else None,
                "p_raw": round(p, 8) if not math.isnan(p) else None,
                "mean_diff": round(float(np.mean(perc_vals[:n] - fixed_vals[:n])), 4),
                "n": n,
            })

    # Holm-Bonferroni correction
    p_vals = [c["p_raw"] if c["p_raw"] is not None else 1.0 for c in comparisons]
    if p_vals:
        adj_ps = holm_bonferroni(p_vals)
        for c, adj_p in zip(comparisons, adj_ps):
            c["p_holm"] = round(adj_p, 8)
            c["significant_holm_05"] = bool(adj_p < 0.05)

    return {"corpus": corpus, "comparisons": comparisons}

# Show pre-computed full-scale statistical tests
stat_tests_full = data['metadata'].get('statistical_tests_cnndm', {})
print("CNN/DM Wilcoxon tests (full-scale, from metadata):")
for c in stat_tests_full.get('comparisons', [])[:6]:  # first 6
    sig = c.get('significant_holm_05', '?')
    print(f"  theta={c['percolation_theta']} vs ratio={c['fixed_ratio']}: "
          f"mean_diff={c.get('mean_diff','?')}, p_holm={c.get('p_holm','?')}, sig={sig}")

# Demo stats (will have insufficient n)
stat_demo = statistical_tests(cnndm_per_doc, corpus="cnndm_demo")
print("\nDemo stats (n=3, expect insufficient n warnings):")
for c in stat_demo['comparisons'][:3]:
    print(f"  theta={c['percolation_theta']} vs ratio={c['fixed_ratio']}: "
          f"mean_diff={c.get('mean_diff','?')}")

## Calibration Analysis

Which theta most closely matches target compression ratios (10%/20%/30%)?

In [ ]:
def calibration_analysis(per_doc: list) -> dict:
    targets = [0.10, 0.20, 0.30]
    result = {}
    for theta in THETAS:
        key = str(theta)
        v = collect_arrays(per_doc, "percolation", key, "compression_ratio")
        if len(v) == 0:
            continue
        mean_cr = float(np.mean(v))
        std_cr = float(np.std(v))
        # Which target is closest?
        closest = min(targets, key=lambda t: abs(mean_cr - t))
        result[f"theta_{key}"] = {
            "mean_compression_ratio": round(mean_cr, 4),
            "std_compression_ratio": round(std_cr, 4),
            "within_doc_std": round(std_cr, 4),
            "closest_target": closest,
            "abs_deviation_from_closest": round(abs(mean_cr - closest), 4),
        }
    # Also evaluate fixed ratios
    for ratio in FIXED_RATIOS:
        key = str(ratio)
        # Fixed ratio always gives ratio * n / n ≈ ratio (small noise from ceil)
        result[f"fixed_{key}"] = {
            "nominal_compression": ratio,
            "comment": "fixed ratio provides near-constant compression by design",
        }
    return result

calib_full = data['metadata'].get('calibration_cnndm', {})
print("Calibration (full-scale CNN/DM):")
for k, v in calib_full.items():
    if 'mean_compression_ratio' in v:
        print(f"  {k}: mean_cr={v['mean_compression_ratio']:.3f}, "
              f"closest_target={v['closest_target']}, "
              f"deviation={v['abs_deviation_from_closest']:.3f}")

calib_demo = calibration_analysis(cnndm_per_doc)
print("\nCalibration (demo, n=3):")
for k, v in calib_demo.items():
    if 'mean_compression_ratio' in v:
        print(f"  {k}: mean_cr={v['mean_compression_ratio']:.3f}")

## Cross-Corpus Comparison

Compare compression ratio variance between CNN/DM and Multi-News at the same thetas via Levene test.

In [ ]:
cross_full = data['metadata'].get('cross_corpus_compression_std', {})
print("Cross-corpus compression std comparison (full-scale):")
print(f"{'Theta':<8} {'CNN/DM std':>12} {'Multi-News std':>15} {'Levene p':>10} {'MN higher var':>14}")
print("-" * 65)
for k, v in cross_full.items():
    print(f"{k:<8} {v['cnndm_compression_std']:>12.4f} {v['multinews_compression_std']:>15.4f} "
          f"{v.get('levene_p', '?'):>10} {str(v['multinews_higher_variance']):>14}")

## TF vs TF-IDF Scorer Comparison (Multi-News)

Does the choice of sentence scorer (TF vs TF-IDF) significantly change ROUGE-1 F1? Wilcoxon test per theta.

In [ ]:
tfidf_full = data['metadata'].get('tfidf_vs_tf_multinews', {})
print("TF vs TF-IDF on Multi-News (full-scale):")
print(f"{'Method':<12} {'TF R1-F':>10} {'TFIDF R1-F':>12} {'Mean diff':>10} {'Wilcoxon p':>12}")
print("-" * 60)
for k, v in tfidf_full.items():
    tf_r = v.get('tf_mean_rouge1_f', '?')
    tfidf_r = v.get('tfidf_mean_rouge1_f', '?')
    diff = v.get('mean_diff_tf_minus_tfidf', '?')
    p = v.get('wilcoxon_p', '?')
    print(f"{k:<12} {tf_r:>10} {tfidf_r:>12} {diff:>10} {str(p):>12}")

## Visualization: Key Results Summary

Plot ROUGE-1 F1 for percolation (all thetas) vs fixed-ratio baselines on CNN/DM, and compression ratio by theta.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Panel 1: ROUGE-1 F1 comparison (full-scale aggregate) ────────────────────
cnndm_agg_full = data['metadata']['cnndm_aggregate']
ax = axes[0]

perc_r1 = [cnndm_agg_full.get(f'percolation_{t}', {}).get('mean_rouge1_f', 0) for t in THETAS]
perc_r1_std = [cnndm_agg_full.get(f'percolation_{t}', {}).get('std_rouge1_f', 0) for t in THETAS]
fixed_r1 = [cnndm_agg_full.get(f'fixed_{r}', {}).get('mean_rouge1_f', 0) for r in FIXED_RATIOS]
fixed_r1_std = [cnndm_agg_full.get(f'fixed_{r}', {}).get('std_rouge1_f', 0) for r in FIXED_RATIOS]

x_perc = [f'θ={t}' for t in THETAS]
x_fixed = [f'{int(r*100)}%' for r in FIXED_RATIOS]

ax.bar(range(len(THETAS)), perc_r1, yerr=perc_r1_std, color='steelblue', alpha=0.8,
       label='Percolation', capsize=4)
ax.bar([len(THETAS) + i for i in range(len(FIXED_RATIOS))], fixed_r1, yerr=fixed_r1_std,
       color='coral', alpha=0.8, label='Fixed ratio', capsize=4)
ax.set_xticks(list(range(len(THETAS))) + [len(THETAS) + i for i in range(len(FIXED_RATIOS))])
ax.set_xticklabels(x_perc + x_fixed, rotation=30, ha='right')
ax.set_ylabel('ROUGE-1 F1')
ax.set_title('CNN/DM: ROUGE-1 F1\n(mean ± std, full-scale n=2000)')
ax.legend()
ax.set_ylim(0, 0.4)

# ── Panel 2: Compression ratio by theta ──────────────────────────────────────
ax = axes[1]
perc_cr = [cnndm_agg_full.get(f'percolation_{t}', {}).get('mean_compression_ratio', 0) for t in THETAS]
perc_cr_std = [cnndm_agg_full.get(f'percolation_{t}', {}).get('std_compression_ratio', 0) for t in THETAS]
ax.errorbar(THETAS, perc_cr, yerr=perc_cr_std, marker='o', color='steelblue',
            linewidth=2, capsize=5, label='Mean compression')
ax.axhline(y=0.10, color='coral', linestyle='--', alpha=0.7, label='10% target')
ax.axhline(y=0.20, color='orange', linestyle='--', alpha=0.7, label='20% target')
ax.axhline(y=0.30, color='green', linestyle='--', alpha=0.7, label='30% target')
ax.set_xlabel('Percolation threshold θ')
ax.set_ylabel('Compression ratio')
ax.set_title('Compression Ratio vs θ\n(CNN/DM, full-scale)')
ax.legend(fontsize=8)

# ── Panel 3: Multi-News TF vs TF-IDF ────────────────────────────────────────
ax = axes[2]
mn_agg_full = data['metadata']['multinews_aggregate']
tf_r1 = [mn_agg_full.get(f'percolation_tf_{t}', {}).get('mean_rouge1_f', 0) for t in THETAS]
tfidf_r1 = [mn_agg_full.get(f'percolation_tfidf_{t}', {}).get('mean_rouge1_f', 0) for t in THETAS]
x = range(len(THETAS))
width = 0.35
ax.bar([xi - width/2 for xi in x], tf_r1, width, color='steelblue', alpha=0.8, label='TF scorer')
ax.bar([xi + width/2 for xi in x], tfidf_r1, width, color='seagreen', alpha=0.8, label='TF-IDF scorer')
ax.set_xticks(list(x))
ax.set_xticklabels([f'θ={t}' for t in THETAS])
ax.set_ylabel('ROUGE-1 F1')
ax.set_title('Multi-News: TF vs TF-IDF Scorer\n(full-scale n=1000)')
ax.legend()

plt.tight_layout()
plt.savefig('results_summary.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to results_summary.png")

## Summary Table

Key findings from the full-scale evaluation (CNN/DM: 2000 docs, Multi-News: 1000 docs).

In [ ]:
m = data['metrics_agg']
print("=" * 60)
print("KEY RESULTS SUMMARY")
print("=" * 60)
print(f"\nCNN/DM ({int(m.get('cnndm_n_docs', 0))} docs):")
print(f"  Best percolation ROUGE-1 F1 (θ=0.6): {m.get('cnndm_perc_0_6_rouge1_f', '?'):.4f}")
print(f"  Best fixed-ratio ROUGE-1 F1 (30%):   {m.get('cnndm_fixed_0_3_rouge1_f', '?'):.4f}")
print(f"  Fixed - Percolation gap:              {m.get('cnndm_fixed_minus_perc_rouge1_f', '?'):.4f}")
print()
print(f"  Compression ratio by theta:")
for t in THETAS:
    safe = str(t).replace('.', '_')
    cr_mean = m.get(f'cnndm_perc_{safe}_compression_mean', '?')
    cr_std = m.get(f'cnndm_perc_{safe}_compression_std', '?')
    ceil_frac = m.get(f'cnndm_perc_{safe}_ceiling_fraction', '?')
    print(f"    θ={t}: mean={cr_mean:.3f}, std={cr_std:.3f}, ceiling_frac={ceil_frac:.3f}")

print()
print(f"Multi-News ({int(m.get('multinews_n_docs', 0))} docs, TF scorer):")
for t in THETAS:
    safe = str(t).replace('.', '_')
    r1 = m.get(f'mn_perc_tf_{safe}_rouge1_f', '?')
    cr_std = m.get(f'mn_perc_tf_{safe}_compression_std', '?')
    print(f"    θ={t}: ROUGE-1 F1={r1:.4f}, compression_std={cr_std:.3f}")

print()
print(f"OLS Regression R² (θ=0.8, CNN/DM): {m.get('cnndm_regression_r2_theta08', '?')}")
print()
print("KEY FINDING: Fixed-ratio baselines consistently dominate ROUGE-1 F1")
print("due to precision advantage. Percolation achieves higher recall but")
print("lower precision. The F1 gap is structural (not scorer-specific).")